# BirdNET baseline — 1024-d embeddings (BirdCLEF 2026)

Experimental notebook: **BirdNET encoder embeddings (1024-D) + small classification heads**, trained on focal `train_audio` and validated on labeled `train_soundscapes` windows (~739 rows).

**Why this path:** We use the BirdNET acoustic encoder API to extract **true 1024-D embeddings** (not classifier logits), then train lightweight heads for BirdCLEF species. Keep this notebook **separate from `src/agent.py`** until you have a headline score.

**Stages (run top to bottom):**

1. Install `birdnet` + scientific stack
2. Paths, species order, multi-label soundscape targets
3. BirdNET embedding helper (**birdnet acoustic encoder**)
4. **Stage 2 a)** Clean cache path: focal + labeled soundscape **feature vectors** → `.npz`
5. **Stage 2 b)** Mix cache path: focal-with-mixing + labeled soundscape **feature vectors** → `.npz`
6. Heads **A** (logistic OvR), **B** (weighted BCE MLP), **C** (same MLP + mixup on features)
7. **Ensemble:** mean of validation probabilities

**Window length:** BirdNET acoustic encoding uses **48 kHz** and **3 s** chunks. We load up to **5 s** at **32 kHz**, resample to **48 kHz**, then center-crop/pad to 3 s before embedding.


In [1]:
import sys
print(sys.version)

import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
# os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# More forcefully disable Metal:
import tensorflow as tf
# tf.config.set_visible_devices([], "GPU")

3.11.15 (main, Mar  3 2026, 00:52:57) [Clang 21.0.0 (clang-2100.0.123.102)]


## Stage 1 — Install

Run once per environment (Python ≥ 3.11). Install **`birdnet`** for true **1024-D acoustic embeddings** plus `tensorflow`, `librosa`, sklearn, etc.

In [2]:
%pip install -q birdnet librosa scipy scikit-learn tensorflow tqdm

Note: you may need to restart the kernel to use updated packages.


## Imports, paths, constants

In [3]:
from __future__ import annotations

import warnings
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.multioutput import MultiOutputClassifier
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)

_HERE = Path.cwd().resolve()
PROJECT_ROOT = _HERE.parent if _HERE.name == "notebooks" else _HERE

DATA_DIR = PROJECT_ROOT / "data"
TRAIN_AUDIO_DIR = DATA_DIR / "train_audio"
TRAIN_SOUNDSCAPES_DIR = DATA_DIR / "train_soundscapes"
TRAIN_CSV = DATA_DIR / "train.csv"
LABELS_CSV = DATA_DIR / "train_soundscapes_labels.csv"
SAMPLE_SUB_CSV = DATA_DIR / "sample_submission.csv"

CACHE_DIR = PROJECT_ROOT / "notebooks" / "birdnet_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
# Separate filenames from older ~6522-D logits caches.
NPZ_CLEAN = CACHE_DIR / "train_embeddings_clean_emb1024.npz"
NPZ_MIX = CACHE_DIR / "train_embeddings_mix_emb1024.npz"
NPZ_VAL = CACHE_DIR / "soundscape_val_embeddings_emb1024.npz"

SR_LOAD = 32000
BIRDNET_SR = 48000
CLIP_LOAD_SEC = 5.0
BIRDNET_CHUNK_SEC = 3.0
# BirdNET acoustic encoder embedding dimension (v2.4 TF backend).
EMB_DIM = 1024

MIX_PROB = 0.35
SNR_MIN_DB, SNR_MAX_DB = 0.0, 15.0
N_MIX_VIEWS = 3

# Encoder throughput knobs (safe defaults for local CPU runs).
EMB_BATCH_SIZE = 32
BIRDNET_N_WORKERS = 4
BIRDNET_N_PRODUCERS = 2

RNG = np.random.default_rng(42)

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: /Users/enricozafiris/Desktop/Catolica/T4 Advanced Predictive Analytics/APA-Project


## Species order + labeled soundscape targets

In [4]:
sample_sub = pd.read_csv(SAMPLE_SUB_CSV)
species_cols = [c for c in sample_sub.columns if c != "row_id"]
species_to_idx = {s: i for i, s in enumerate(species_cols)}
n_species = len(species_cols)
print("n_species:", n_species)

train_df = pd.read_csv(TRAIN_CSV)


def _tok(v) -> set[str]:
    if pd.isna(v) or v == "":
        return set()
    return {t.strip() for t in str(v).split(";") if t.strip()}


def _merge_labels(series: pd.Series) -> set[str]:
    o: set[str] = set()
    for v in series:
        o |= _tok(v)
    return o


lab = pd.read_csv(LABELS_CSV)
grp = (
    lab.groupby(["filename", "start", "end"], sort=False)["primary_label"]
    .agg(_merge_labels)
    .reset_index()
)
grp["end_sec"] = pd.to_timedelta(grp["end"]).dt.total_seconds().astype(int)
grp["row_id"] = grp["filename"].str.replace(".ogg", "", regex=False) + "_" + grp["end_sec"].astype(str)

y_true_rows = []
unknown_codes: set[str] = set()
for _, r in grp.iterrows():
    v = np.zeros(n_species, dtype=np.float32)
    for code in r["primary_label"]:
        j = species_to_idx.get(code)
        if j is None:
            unknown_codes.add(code)
        else:
            v[j] = 1.0
    y_true_rows.append((r["row_id"], v))

if unknown_codes:
    print("WARNING: label tokens missing from sample_submission:", len(unknown_codes))

y_true_df = pd.DataFrame(
    [v for _, v in y_true_rows], index=[rid for rid, _ in y_true_rows], columns=species_cols
).sort_index()
if not y_true_df.index.is_unique:
    y_true_df = y_true_df.groupby(level=0).max()

labeled_stems = {rid.rsplit("_", 1)[0] for rid in y_true_df.index}
print("Labeled soundscape windows:", len(y_true_df))

n_species: 234
Labeled soundscape windows: 739


## Stage 1 — BirdNET embedding helper

We call the **BirdNET acoustic encoder API** (`birdnet.load(...).encode_session(...)`) to extract true **1024-D embeddings**.

Audio: resample/pad to one **3 s** chunk at **48 kHz**, then extract one embedding vector.

In [5]:
import threading

import birdnet

_EMB_LOCK = threading.Lock()
_bird_model = None
_bird_session = None
_bird_sr = None


def _init_birdnet_encoder() -> None:
    global _bird_model, _bird_session, _bird_sr
    if _bird_session is not None:
        return
    _bird_model = birdnet.load("acoustic", "2.4", "tf")
    _bird_sr = int(_bird_model.get_sample_rate())
    _bird_session = _bird_model.encode_session(
        batch_size=EMB_BATCH_SIZE,
        prefetch_ratio=2,
        n_workers=BIRDNET_N_WORKERS,
        n_producers=BIRDNET_N_PRODUCERS,
    )
    _bird_session.__enter__()
    print(
        "BirdNET encoder ready | sample_rate=",
        _bird_sr,
        "| emb_dim=",
        EMB_DIM,
        "| batch_size=",
        EMB_BATCH_SIZE,
        "| workers=",
        BIRDNET_N_WORKERS,
        "| producers=",
        BIRDNET_N_PRODUCERS,
    )


def audio_to_birdnet_chunk(y: np.ndarray, sr: int) -> np.ndarray:
    y = np.asarray(y, dtype=np.float32).reshape(-1)
    if sr != BIRDNET_SR:
        y = librosa.resample(y, orig_sr=sr, target_sr=BIRDNET_SR)
    max_keep = int(CLIP_LOAD_SEC * BIRDNET_SR)
    if len(y) > max_keep:
        y = y[:max_keep]
    need = int(BIRDNET_CHUNK_SEC * BIRDNET_SR)
    if len(y) >= need:
        s0 = max(0, (len(y) - need) // 2)
        y = y[s0 : s0 + need]
    else:
        y = np.pad(y, (0, need - len(y)))
    return y.astype(np.float32)


def embed_audio_batch(batch_audio: list[np.ndarray], sr: int) -> np.ndarray:
    if not batch_audio:
        return np.empty((0, EMB_DIM), dtype=np.float32)
    batch_inputs = [
        (np.ascontiguousarray(audio_to_birdnet_chunk(y, sr), dtype=np.float32), BIRDNET_SR)
        for y in batch_audio
    ]
    with _EMB_LOCK:
        _init_birdnet_encoder()
        assert _bird_session is not None
        res = _bird_session.run_arrays(batch_inputs)
    emb = np.asarray(res.embeddings[:, 0, :], dtype=np.float32)
    if emb.ndim != 2 or emb.shape[1] != EMB_DIM:
        raise RuntimeError(f"Unexpected BirdNET embedding tensor shape {emb.shape}; expected (*, {EMB_DIM})")
    return emb


def embed_audio_mono(y: np.ndarray, sr: int, *, analyzer=None) -> np.ndarray:
    _ = analyzer
    return embed_audio_batch([y], sr)[0]


def load_focal_clip(path: Path) -> tuple[np.ndarray, int]:
    y, sr = librosa.load(str(path), sr=SR_LOAD, mono=True)
    n = int(CLIP_LOAD_SEC * SR_LOAD)
    if len(y) > n:
        y = y[:n]
    elif len(y) < n:
        y = np.pad(y, (0, n - len(y)))
    return y.astype(np.float32), SR_LOAD

## Stage 2 a) — Cache embeddings (clean focal + labeled windows)

- Independent path A: run this for clean focal training cache + validation cache.
- Set `FORCE_REBUILD = True` to overwrite caches.
- Set `MAX_FOCAL = 2000` (for example) for a fast smoke test.
- Skips missing audio paths silently.

In [ ]:
FORCE_REBUILD = False
MAX_FOCAL: int | None = None


def macro_auc_ge3(y_true: np.ndarray, y_score: np.ndarray) -> tuple[float, int, int]:
    pos = y_true.sum(axis=0)
    keep = pos >= 3
    if not np.any(keep):
        return float("nan"), 0, int(np.sum(pos > 0))
    yt = y_true[:, keep]
    ys = y_score[:, keep]
    usable = []
    for j in range(yt.shape[1]):
        col = yt[:, j]
        if col.min() == 0 and col.max() == 1:
            usable.append(j)
    if not usable:
        return float("nan"), int(np.sum(keep)), int(np.sum(pos > 0))
    usable = np.array(usable, dtype=int)
    auc = roc_auc_score(yt[:, usable], ys[:, usable], average="macro")
    return float(auc), int(np.sum(keep)), int(np.sum(pos > 0))


def build_focal_manifest() -> pd.DataFrame:
    rows = []
    for _, r in train_df.iterrows():
        lab = str(r["primary_label"])
        rel = r["filename"]
        if lab not in species_to_idx:
            continue
        ap = TRAIN_AUDIO_DIR / str(rel)
        if not ap.is_file():
            continue
        rows.append({"path": str(ap.resolve()), "primary_label": lab})
    mf = pd.DataFrame(rows)
    if MAX_FOCAL is not None:
        mf = mf.head(int(MAX_FOCAL))
    return mf


def soundscape_window_audio(row_id: str) -> tuple[np.ndarray, int]:
    stem, end_s = row_id.rsplit("_", 1)
    end_sec = int(end_s)
    start_sec = max(0, end_sec - int(CLIP_LOAD_SEC))
    fp = TRAIN_SOUNDSCAPES_DIR / f"{stem}.ogg"
    y, _ = librosa.load(
        str(fp), sr=SR_LOAD, mono=True, offset=start_sec, duration=CLIP_LOAD_SEC
    )
    n = int(CLIP_LOAD_SEC * SR_LOAD)
    if len(y) > n:
        y = y[:n]
    elif len(y) < n:
        y = np.pad(y, (0, n - len(y)))
    return y.astype(np.float32), SR_LOAD


if FORCE_REBUILD or not NPZ_CLEAN.is_file() or not NPZ_VAL.is_file():
    manifest = build_focal_manifest()
    print("Focal rows to embed:", len(manifest))

    X_list, y_list, p_list = [], [], []
    for start in tqdm(range(0, len(manifest), EMB_BATCH_SIZE), desc="Stage2a focal", total=(len(manifest) + EMB_BATCH_SIZE - 1) // EMB_BATCH_SIZE):
        chunk_rows = manifest.iloc[start : start + EMB_BATCH_SIZE]
        audios, labels, paths = [], [], []
        for r in chunk_rows.itertuples(index=False):
            y_wav, _ = load_focal_clip(Path(r.path))
            audios.append(y_wav)
            vec = np.zeros(n_species, dtype=np.float32)
            vec[species_to_idx[r.primary_label]] = 1.0
            labels.append(vec)
            paths.append(r.path)
        embs = embed_audio_batch(audios, SR_LOAD)
        X_list.extend([e for e in embs])
        y_list.extend(labels)
        p_list.extend(paths)

    X_tr = np.stack(X_list, axis=0)
    y_tr = np.stack(y_list, axis=0)

    valid_rids = []
    val_audios = []
    for rid in list(y_true_df.index):
        stem = rid.rsplit("_", 1)[0]
        if not (TRAIN_SOUNDSCAPES_DIR / f"{stem}.ogg").is_file():
            continue
        y_wav, _ = soundscape_window_audio(rid)
        valid_rids.append(rid)
        val_audios.append(y_wav)

    X_va_parts = []
    for start in tqdm(range(0, len(val_audios), EMB_BATCH_SIZE), desc="Stage2a val", total=(len(val_audios) + EMB_BATCH_SIZE - 1) // EMB_BATCH_SIZE):
        X_va_parts.append(embed_audio_batch(val_audios[start : start + EMB_BATCH_SIZE], SR_LOAD))
    X_va = np.concatenate(X_va_parts, axis=0)
    rid_va = np.array(valid_rids, dtype=object)
    y_va = y_true_df.loc[rid_va].to_numpy(dtype=np.float32)

    np.savez_compressed(NPZ_CLEAN, X_train=X_tr, y_train=y_tr, train_paths=np.array(p_list, dtype=object))
    np.savez_compressed(NPZ_VAL, X_val=X_va, row_ids=rid_va, y_val=y_va)
    print("Saved", NPZ_CLEAN, X_tr.shape)
    print("Saved", NPZ_VAL, X_va.shape)
else:
    d1 = np.load(NPZ_CLEAN, allow_pickle=True)
    d2 = np.load(NPZ_VAL, allow_pickle=True)
    print("Loaded", NPZ_CLEAN, d1["X_train"].shape, "|", NPZ_VAL, d2["X_val"].shape)

## Stage 2 b) — Mix focal cache + validation cache

Builds mixed focal training features (for `USE_MIX_TRAIN=True`) by sampling backgrounds from **`train_soundscapes` files whose stems are not in the labeled evaluation set** and mixing at random SNR.

Also ensures `soundscape_val_embeddings_emb1024.npz` exists, so this stage can be run independently of Stage 2 a) for the mix-training path.

Cost: training cache is roughly **`× N_MIX_VIEWS`** inference work versus clean focal caching.

In [6]:
FORCE_REBUILD_MIX = True
FORCE_REBUILD_VAL_FROM_STAGE4 = False

noise_candidates = sorted(TRAIN_SOUNDSCAPES_DIR.glob("*.ogg"))
noise_pool = [p for p in noise_candidates if p.stem not in labeled_stems]
print("Noise pool files:", len(noise_pool))


def random_soundscape_noise_5s() -> np.ndarray:
    if not noise_pool:
        raise RuntimeError("No soundscapes available for mixing.")
    fp = Path(noise_pool[int(RNG.integers(0, len(noise_pool)))])
    dur = librosa.get_duration(path=str(fp))
    start_max = max(0.0, dur - CLIP_LOAD_SEC)
    offset = float(RNG.uniform(0, start_max)) if start_max > 0 else 0.0
    n, _sr = librosa.load(str(fp), sr=SR_LOAD, mono=True, offset=offset, duration=CLIP_LOAD_SEC)
    nt = int(CLIP_LOAD_SEC * SR_LOAD)
    if len(n) > nt:
        n = n[:nt]
    elif len(n) < nt:
        n = np.pad(n, (0, nt - len(n)))
    return n.astype(np.float32)


def mix_snr(signal: np.ndarray, noise: np.ndarray, snr_db: float) -> np.ndarray:
    s = signal.astype(np.float32)
    n = noise.astype(np.float32)
    ps = np.mean(s ** 2) + 1e-12
    pn = np.mean(n ** 2) + 1e-12
    scale = np.sqrt(ps / (pn * (10 ** (snr_db / 10.0))))
    return np.clip(s + scale * n, -1.0, 1.0)


def build_focal_manifest_stage4() -> pd.DataFrame:
    rows = []
    for _, r in train_df.iterrows():
        lab = str(r["primary_label"])
        rel = r["filename"]
        if lab not in species_to_idx:
            continue
        ap = TRAIN_AUDIO_DIR / str(rel)
        if not ap.is_file():
            continue
        rows.append({"path": str(ap.resolve()), "primary_label": lab})
    return pd.DataFrame(rows)


def soundscape_window_audio_stage4(row_id: str) -> tuple[np.ndarray, int]:
    stem, end_s = row_id.rsplit("_", 1)
    end_sec = int(end_s)
    start_sec = max(0, end_sec - int(CLIP_LOAD_SEC))
    fp = TRAIN_SOUNDSCAPES_DIR / f"{stem}.ogg"
    y, _ = librosa.load(str(fp), sr=SR_LOAD, mono=True, offset=start_sec, duration=CLIP_LOAD_SEC)
    n = int(CLIP_LOAD_SEC * SR_LOAD)
    if len(y) > n:
        y = y[:n]
    elif len(y) < n:
        y = np.pad(y, (0, n - len(y)))
    return y.astype(np.float32), SR_LOAD



Noise pool files: 10592


In [ ]:

if FORCE_REBUILD_MIX or not NPZ_MIX.is_file():
    manifest = build_focal_manifest_stage4()
    Xmx, ymx, meta_path, meta_vid = [], [], [], []

    total_mix_rows = len(manifest) * N_MIX_VIEWS
    pbar = tqdm(total=total_mix_rows, desc="Stage4 mix")
    for start in range(0, len(manifest), EMB_BATCH_SIZE):
        chunk_rows = manifest.iloc[start : start + EMB_BATCH_SIZE]
        mix_audios, mix_labels, mix_paths, mix_variants = [], [], [], []

        for r in chunk_rows.itertuples(index=False):
            y_clean, _ = load_focal_clip(Path(r.path))
            lbl_vec = np.zeros(n_species, dtype=np.float32)
            lbl_vec[species_to_idx[r.primary_label]] = 1.0

            for v in range(N_MIX_VIEWS):
                if v > 0 and noise_pool and RNG.random() < MIX_PROB:
                    noise = random_soundscape_noise_5s()
                    snr = float(RNG.uniform(SNR_MIN_DB, SNR_MAX_DB))
                    y_use = mix_snr(y_clean, noise, snr)
                else:
                    y_use = y_clean
                mix_audios.append(y_use)
                mix_labels.append(lbl_vec.copy())
                mix_paths.append(r.path)
                mix_variants.append(np.int16(v))
                pbar.update(1)

        embs = embed_audio_batch(mix_audios, SR_LOAD)
        Xmx.extend([e for e in embs])
        ymx.extend(mix_labels)
        meta_path.extend(mix_paths)
        meta_vid.extend(mix_variants)

    pbar.close()

    np.savez_compressed(
        NPZ_MIX,
        X_train=np.stack(Xmx),
        y_train=np.stack(ymx),
        paths=np.array(meta_path, dtype=object),
        variant=np.array(meta_vid),
    )
    print("Saved", NPZ_MIX, np.stack(Xmx).shape)
else:
    print("Mix cache exists:", NPZ_MIX)


if FORCE_REBUILD_VAL_FROM_STAGE4 or not NPZ_VAL.is_file():
    valid_rids, val_audios = [], []
    for rid in list(y_true_df.index):
        stem = rid.rsplit("_", 1)[0]
        if not (TRAIN_SOUNDSCAPES_DIR / f"{stem}.ogg").is_file():
            continue
        y_wav, _ = soundscape_window_audio_stage4(rid)
        valid_rids.append(rid)
        val_audios.append(y_wav)

    X_va_parts = []
    for start in tqdm(range(0, len(val_audios), EMB_BATCH_SIZE), desc="Stage4 val cache", total=(len(val_audios) + EMB_BATCH_SIZE - 1) // EMB_BATCH_SIZE):
        X_va_parts.append(embed_audio_batch(val_audios[start : start + EMB_BATCH_SIZE], SR_LOAD))
    X_va = np.concatenate(X_va_parts, axis=0)
    rid_va = np.array(valid_rids, dtype=object)
    y_va = y_true_df.loc[rid_va].to_numpy(dtype=np.float32)
    np.savez_compressed(NPZ_VAL, X_val=X_va, row_ids=rid_va, y_val=y_va)
    print("Saved", NPZ_VAL, X_va.shape)
else:
    d2 = np.load(NPZ_VAL, allow_pickle=True)
    print("Val cache exists:", NPZ_VAL, d2["X_val"].shape)

## 2 c Random Augmentation Mini Experiments

In [31]:
# ─────────────────────────────────────────────────────────────────────────
# Stage 2 c) — Augmentation strength sweep on subset
# ─────────────────────────────────────────────────────────────────────────
# Trains Head B on a small stratified subset of focal data with several
# augmentation presets, validates against the full labeled soundscape pool.
# Goal: cheap exploration of which augmentation strength produces the best
# soundscape AUC, before committing to a full-data rebuild.
#
# First preset replicates your existing 2b settings exactly so you can see
# how close the subset-AUC is to your full-data AUC (~0.69).

import time
import json
from typing import Optional

# ── Sweep configuration ──────────────────────────────────────────────────
SUBSET_FOCAL_SIZE = 2000       # stratified subset size for sweep
MIN_PER_SPECIES = 3            # minimum clips per species in subset
SWEEP_EPOCHS = 30
SWEEP_PATIENCE = 5
SWEEP_BATCH_SIZE = 256

# Each preset is a full augmentation config. Numbers are intentionally
# spread across a wide range so weak/strong differences are visible.
SWEEP_PRESETS = [
    # 0: replicate current 2b — calibration anchor
    {"name": "current_2b",   "MIX_PROB": 0.35, "SNR_MIN": 0.0,  "SNR_MAX": 15.0,
     "N_VIEWS": 3, "TIME_SHIFT_S": 0.0, "GAIN_DB": 0.0},
    # 1: very light — almost clean, small perturbations
    {"name": "very_light",   "MIX_PROB": 0.20, "SNR_MIN": 10.0, "SNR_MAX": 20.0,
     "N_VIEWS": 2, "TIME_SHIFT_S": 0.3, "GAIN_DB": 1.5},
    # 2: light
    {"name": "light",        "MIX_PROB": 0.40, "SNR_MIN": 5.0,  "SNR_MAX": 18.0,
     "N_VIEWS": 3, "TIME_SHIFT_S": 0.5, "GAIN_DB": 3.0},
    # 3: medium
    {"name": "medium",       "MIX_PROB": 0.65, "SNR_MIN": 0.0,  "SNR_MAX": 12.0,
     "N_VIEWS": 3, "TIME_SHIFT_S": 0.8, "GAIN_DB": 4.0},
    # 4: strong
    {"name": "strong",       "MIX_PROB": 0.85, "SNR_MIN": -3.0, "SNR_MAX": 10.0,
     "N_VIEWS": 4, "TIME_SHIFT_S": 1.2, "GAIN_DB": 5.0},
    # 5: very strong — close to test-distribution noise levels
    {"name": "very_strong",  "MIX_PROB": 0.95, "SNR_MIN": -5.0, "SNR_MAX": 8.0,
     "N_VIEWS": 5, "TIME_SHIFT_S": 1.5, "GAIN_DB": 6.0},
]

SWEEP_OUT_DIR = AUTOSAVE_DIR / "sweep_2c"
SWEEP_OUT_DIR.mkdir(parents=True, exist_ok=True)
SWEEP_RESULTS_PATH = SWEEP_OUT_DIR / "results.json"


# ── Stratified subset builder ────────────────────────────────────────────
def build_stratified_subset(manifest: pd.DataFrame, n_target: int, min_per_species: int,
                             rng: np.random.Generator) -> pd.DataFrame:
    """Pick ~n_target rows: at least min_per_species per species when available, then random fill."""
    by_sp = {sp: g.reset_index(drop=True) for sp, g in manifest.groupby("primary_label")}
    picked_idx = []
    leftover = []
    for sp, g in by_sp.items():
        idx_pool = list(g.index)
        rng.shuffle(idx_pool)
        take = min(len(idx_pool), min_per_species)
        for i in idx_pool[:take]:
            picked_idx.append((sp, int(i)))
        for i in idx_pool[take:]:
            leftover.append((sp, int(i)))
    rng.shuffle(leftover)
    needed = max(0, n_target - len(picked_idx))
    picked_idx.extend(leftover[:needed])
    rows = [by_sp[sp].iloc[i] for sp, i in picked_idx]
    return pd.DataFrame(rows).reset_index(drop=True).head(n_target)


# ── Augmentation primitives ──────────────────────────────────────────────
def _time_shift(y: np.ndarray, max_shift_s: float, sr: int, rng: np.random.Generator) -> np.ndarray:
    if max_shift_s <= 0.0:
        return y
    n = int(rng.uniform(-max_shift_s, max_shift_s) * sr)
    return np.roll(y, n)

def _gain_jitter(y: np.ndarray, max_db: float, rng: np.random.Generator) -> np.ndarray:
    if max_db <= 0.0:
        return y
    db = float(rng.uniform(-max_db, max_db))
    return (y * (10 ** (db / 20.0))).astype(np.float32)


# ── Build features for one preset ────────────────────────────────────────
def build_subset_features(subset_df: pd.DataFrame, preset: dict, rng: np.random.Generator):
    """Embed the subset with this preset's augmentation. Returns X, y arrays."""
    X_list, y_list = [], []
    n_views = int(preset["N_VIEWS"])

    for r in tqdm(subset_df.itertuples(index=False), total=len(subset_df),
                   desc=f"  embed {preset['name']}", leave=False):
        try:
            y_clean, sr = load_focal_clip(Path(r.path))
        except Exception:
            continue
        lbl = np.zeros(n_species, dtype=np.float32)
        lbl[species_to_idx[r.primary_label]] = 1.0

        for v in range(n_views):
            y_use = y_clean
            # First view is always clean (anchor); subsequent views may be augmented
            if v > 0:
                y_use = _time_shift(y_use, preset["TIME_SHIFT_S"], sr, rng)
                y_use = _gain_jitter(y_use, preset["GAIN_DB"], rng)
                if noise_pool and rng.random() < preset["MIX_PROB"]:
                    noise = random_soundscape_noise_5s()
                    snr = float(rng.uniform(preset["SNR_MIN"], preset["SNR_MAX"]))
                    y_use = mix_snr(y_use, noise, snr)
            try:
                emb = embed_audio_mono(y_use, sr)
            except Exception:
                continue
            X_list.append(emb)
            y_list.append(lbl.copy())

    if not X_list:
        return None, None
    return np.stack(X_list), np.stack(y_list)


# ── Train + evaluate Head B for one preset ───────────────────────────────
def train_eval_head(X_tr: np.ndarray, y_tr: np.ndarray,
                     X_va: np.ndarray, y_va: np.ndarray) -> dict:
    """Train a fresh Head B on (X_tr, y_tr), return validation metrics."""
    pos = y_tr.sum(axis=0).astype(np.float64)
    neg = len(y_tr) - pos
    pw = np.clip(neg / np.maximum(pos, 1.0), 1.0, 25.0).astype(np.float32)
    pw_t = tf.constant(pw)[tf.newaxis, :]

    def loss_fn(yt, yp):
        yp = tf.clip_by_value(yp, 1e-7, 1.0 - 1e-7)
        return tf.reduce_mean(
            pw_t * yt * (-tf.math.log(yp))
            + (1.0 - yt) * (-tf.math.log(1.0 - yp)),
            axis=-1,
        )

    tf.keras.utils.set_random_seed(42)
    inp = tf.keras.layers.Input(shape=(EMB_DIM,))
    h = tf.keras.layers.Dense(512, activation="relu")(inp)
    h = tf.keras.layers.Dropout(0.3)(h)
    out = tf.keras.layers.Dense(n_species, activation="sigmoid")(h)
    model = tf.keras.Model(inp, out)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=loss_fn)

    cb = tf.keras.callbacks.EarlyStopping(
        patience=SWEEP_PATIENCE, restore_best_weights=True, monitor="val_loss",
    )
    t0 = time.time()
    hist = model.fit(
        X_tr, y_tr,
        validation_split=0.1,
        epochs=SWEEP_EPOCHS,
        batch_size=SWEEP_BATCH_SIZE,
        callbacks=[cb],
        verbose=0,
    )
    train_secs = time.time() - t0

    pred = model.predict(X_va, batch_size=512, verbose=0)
    auc, k3, k1 = macro_auc_ge3(y_va, pred)
    epochs_run = len(hist.history["loss"])
    return {
        "auc_ge3": float(auc),
        "k3": int(k3),
        "k1": int(k1),
        "epochs_run": epochs_run,
        "train_secs": round(train_secs, 1),
        "n_train_features": int(len(X_tr)),
    }


# ── Run the sweep ────────────────────────────────────────────────────────
print("=" * 70)
print(f"Stage 2c sweep: {len(SWEEP_PRESETS)} presets x subset of {SUBSET_FOCAL_SIZE} focal clips")
print(f"Validation: full labeled soundscape pool ({len(y_true_df)} windows)")
print("=" * 70)

# Load val cache once
dval = np.load(NPZ_VAL, allow_pickle=True)
X_val_sweep = dval["X_val"].astype(np.float32)
y_val_sweep = dval["y_val"].astype(np.float32)

# Build manifest + stratified subset (same across presets for fair comparison)
sweep_rng = np.random.default_rng(42)
manifest_full = build_focal_manifest_stage4()
subset_df = build_stratified_subset(
    manifest_full, n_target=SUBSET_FOCAL_SIZE, min_per_species=MIN_PER_SPECIES, rng=sweep_rng,
)
print(f"Subset built: {len(subset_df)} clips across "
      f"{subset_df['primary_label'].nunique()} species\n")

results = []
for preset in SWEEP_PRESETS:
    print(f"── Preset: {preset['name']} | {preset}")
    preset_rng = np.random.default_rng(hash(preset["name"]) & 0xFFFFFFFF)
    t0 = time.time()
    X_tr, y_tr = build_subset_features(subset_df, preset, preset_rng)
    if X_tr is None or len(X_tr) == 0:
        print(f"   ✗ no features built; skipping")
        continue
    embed_secs = time.time() - t0
    print(f"   features: X={X_tr.shape}  embed_time={embed_secs:.1f}s")

    metrics = train_eval_head(X_tr, y_tr, X_val_sweep, y_val_sweep)
    metrics["embed_secs"] = round(embed_secs, 1)
    metrics["preset"] = preset
    results.append(metrics)

    print(f"   AUC≥3 = {metrics['auc_ge3']:.4f}  "
          f"(k3={metrics['k3']}, epochs_run={metrics['epochs_run']}, "
          f"train={metrics['train_secs']}s)\n")
    # Persist after every preset so you don't lose progress
    SWEEP_RESULTS_PATH.write_text(json.dumps(results, indent=2))

# ── Summary ──────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("SWEEP SUMMARY (sorted by AUC≥3)")
print("=" * 70)
results_sorted = sorted(results, key=lambda r: r["auc_ge3"], reverse=True)
for r in results_sorted:
    p = r["preset"]
    print(f"  {p['name']:14s} AUC≥3={r['auc_ge3']:.4f} k3={r['k3']:3d}  "
          f"MIX={p['MIX_PROB']:.2f} SNR=[{p['SNR_MIN']:+.1f},{p['SNR_MAX']:+.1f}] "
          f"V={p['N_VIEWS']} TS={p['TIME_SHIFT_S']:.1f}s G={p['GAIN_DB']:.1f}dB")

# Calibration anchor: how does the current_2b subset AUC compare to your full-data AUC?
anchor = next((r for r in results if r["preset"]["name"] == "current_2b"), None)
if anchor:
    print(f"\nCalibration: current_2b on subset → AUC≥3={anchor['auc_ge3']:.4f}")
    print(f"             your full-data 2b head → AUC ≈ 0.69")
    print(f"             gap = {0.69 - anchor['auc_ge3']:+.3f} "
          f"(if large, subset is too small for reliable ranking)")

print(f"\nResults saved: {SWEEP_RESULTS_PATH}")
print("Inspect, pick a winner, set WINNER_PRESET below, then run the next cell.")

Stage 2c sweep: 6 presets x subset of 2000 focal clips
Validation: full labeled soundscape pool (739 windows)
Subset built: 2000 clips across 206 species

── Preset: current_2b | {'name': 'current_2b', 'MIX_PROB': 0.35, 'SNR_MIN': 0.0, 'SNR_MAX': 15.0, 'N_VIEWS': 3, 'TIME_SHIFT_S': 0.0, 'GAIN_DB': 0.0}


  embed current_2b:   0%|          | 0/2000 [00:00<?, ?it/s]

BirdNET encoder ready | sample_rate= 48000 | emb_dim= 1024 | batch_size= 32 | workers= 4 | producers= 2


Process 6d540-Producer-0:
Process 6d540-Producer-1:
Process 6d540-Worker-0:
Process 6d540-Worker-1:
Process 6d540-Worker-2:
Process 6d540-Worker-3:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.11/3.11.15_1/Frameworks/Python.framework/Versions/3.11/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/opt/homebrew/Cellar/python@3.11/3.11.15_1/Frameworks/Python.framework/Versions/3.11/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/enricozafiris/Desktop/Catolica/T4 Advanced Predictive Analytics/APA-Project/.venv311/lib/python3.11/site-packages/birdnet/acoustic/inference/core/worker.py", line 195, in __call__
    self.run_main_loop()
  File "/Users/enricozafiris/Desktop/Catolica/

KeyboardInterrupt: 

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# Stage 2 d) — Full rebuild with winning preset + retrain Head B
# ─────────────────────────────────────────────────────────────────────────
# Takes the chosen preset from 2c, rebuilds the focal+mix cache on ALL data,
# retrains Head B end-to-end, and saves a drop-in replacement bundle for
# your existing submission flow.

# ── Pick your winner ─────────────────────────────────────────────────────
# Either set WINNER_PRESET = "current_2b" (or whichever name from 2c results),
# or paste a custom preset dict directly to override.
WINNER_PRESET_NAME: Optional[str] = "current_2b"
WINNER_PRESET_OVERRIDE: Optional[dict] = None  # set to a dict to override

# Final-rebuild knobs
RETRAIN_EPOCHS = 80
RETRAIN_PATIENCE = 5
RETRAIN_BATCH_SIZE = 256

FINAL_OUT_DIR = AUTOSAVE_DIR / "final_2d"
FINAL_OUT_DIR.mkdir(parents=True, exist_ok=True)
NPZ_FINAL = FINAL_OUT_DIR / "train_embeddings_final.npz"

# Resolve the chosen preset
if WINNER_PRESET_OVERRIDE is not None:
    chosen = dict(WINNER_PRESET_OVERRIDE)
    chosen.setdefault("name", "custom")
else:
    sweep_results = json.loads(SWEEP_RESULTS_PATH.read_text())
    by_name = {r["preset"]["name"]: r["preset"] for r in sweep_results}
    if WINNER_PRESET_NAME not in by_name:
        raise KeyError(f"Preset '{WINNER_PRESET_NAME}' not in sweep results: {list(by_name)}")
    chosen = by_name[WINNER_PRESET_NAME]

print(f"Chosen preset: {chosen}\n")

# ── Build full mix cache ─────────────────────────────────────────────────
manifest_all = build_focal_manifest_stage4()
print(f"Building full mix cache: {len(manifest_all)} focal files × {chosen['N_VIEWS']} views")

final_rng = np.random.default_rng(2026)
Xmx, ymx, paths_meta, var_meta = [], [], [], []
n_views = int(chosen["N_VIEWS"])

for r in tqdm(manifest_all.itertuples(index=False), total=len(manifest_all), desc="final mix"):
    try:
        y_clean, sr = load_focal_clip(Path(r.path))
    except Exception:
        continue
    lbl = np.zeros(n_species, dtype=np.float32)
    lbl[species_to_idx[r.primary_label]] = 1.0

    for v in range(n_views):
        y_use = y_clean
        if v > 0:
            y_use = _time_shift(y_use, chosen["TIME_SHIFT_S"], sr, final_rng)
            y_use = _gain_jitter(y_use, chosen["GAIN_DB"], final_rng)
            if noise_pool and final_rng.random() < chosen["MIX_PROB"]:
                noise = random_soundscape_noise_5s()
                snr = float(final_rng.uniform(chosen["SNR_MIN"], chosen["SNR_MAX"]))
                y_use = mix_snr(y_use, noise, snr)
        try:
            emb = embed_audio_mono(y_use, sr)
        except Exception:
            continue
        Xmx.append(emb)
        ymx.append(lbl.copy())
        paths_meta.append(r.path)
        var_meta.append(np.int16(v))

X_full = np.stack(Xmx).astype(np.float32)
y_full = np.stack(ymx).astype(np.float32)
np.savez_compressed(
    NPZ_FINAL,
    X_train=X_full, y_train=y_full,
    paths=np.array(paths_meta, dtype=object),
    variant=np.array(var_meta),
    preset=np.array([json.dumps(chosen)], dtype=object),
)
print(f"\nSaved full cache: {NPZ_FINAL} | X={X_full.shape}")

# ── Retrain Head B on full cache ─────────────────────────────────────────
pos = y_full.sum(axis=0).astype(np.float64)
neg = len(y_full) - pos
pw = np.clip(neg / np.maximum(pos, 1.0), 1.0, 25.0).astype(np.float32)
pw_t = tf.constant(pw)[tf.newaxis, :]

def final_loss_fn(yt, yp):
    yp = tf.clip_by_value(yp, 1e-7, 1.0 - 1e-7)
    return tf.reduce_mean(
        pw_t * yt * (-tf.math.log(yp))
        + (1.0 - yt) * (-tf.math.log(1.0 - yp)),
        axis=-1,
    )

tf.keras.utils.set_random_seed(42)
inp = tf.keras.layers.Input(shape=(EMB_DIM,))
h = tf.keras.layers.Dense(512, activation="relu")(inp)
h = tf.keras.layers.Dropout(0.3)(h)
out = tf.keras.layers.Dense(n_species, activation="sigmoid")(h)
final_model = tf.keras.Model(inp, out)
final_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=final_loss_fn)

print(f"\nRetraining Head B: {len(X_full)} samples, max {RETRAIN_EPOCHS} epochs")
cb = tf.keras.callbacks.EarlyStopping(
    patience=RETRAIN_PATIENCE, restore_best_weights=True, monitor="val_loss",
)
t0 = time.time()
final_model.fit(
    X_full, y_full,
    validation_split=0.1,
    epochs=RETRAIN_EPOCHS,
    batch_size=RETRAIN_BATCH_SIZE,
    callbacks=[cb],
    verbose=1,
)
print(f"Train time: {time.time()-t0:.1f}s")

# ── Evaluate on labeled soundscape val ───────────────────────────────────
dval = np.load(NPZ_VAL, allow_pickle=True)
X_val_eval = dval["X_val"].astype(np.float32)
y_val_eval = dval["y_val"].astype(np.float32)
final_pred = final_model.predict(X_val_eval, batch_size=512, verbose=0)
final_auc, final_k3, final_k1 = macro_auc_ge3(y_val_eval, final_pred)
print(f"\nFINAL Head B macro-AUC≥3 = {final_auc:.4f}  (k3={final_k3}, k1={final_k1})")

# ── Save bundle (drop-in replacement for submission) ─────────────────────
final_model.save(FINAL_OUT_DIR / "head_b_mlp.keras")
final_model.save_weights(FINAL_OUT_DIR / "head_b_weights.weights.h5")

# Reuse the metadata from the bundle dir if it exists; else recreate
src_meta = AUTOSAVE_DIR / "metadata.npz"
dst_meta = FINAL_OUT_DIR / "metadata.npz"
if src_meta.is_file():
    import shutil
    shutil.copy2(src_meta, dst_meta)
else:
    np.savez_compressed(
        dst_meta,
        species_cols=np.array(species_cols, dtype=object),
        emb_dim=np.int32(EMB_DIM),
        sr_load=np.int32(SR_LOAD),
        birdnet_sr=np.int32(BIRDNET_SR),
        clip_load_sec=np.float32(CLIP_LOAD_SEC),
        birdnet_chunk_sec=np.float32(BIRDNET_CHUNK_SEC),
    )

# Persist the preset that produced this model
(FINAL_OUT_DIR / "preset.json").write_text(json.dumps(chosen, indent=2))
(FINAL_OUT_DIR / "metrics.json").write_text(json.dumps({
    "auc_ge3": float(final_auc), "k3": int(final_k3), "k1": int(final_k1),
    "n_train": int(len(X_full)), "preset": chosen,
}, indent=2))

print(f"\nSaved bundle: {FINAL_OUT_DIR}")
print(f"  head_b_mlp.keras")
print(f"  head_b_weights.weights.h5")
print(f"  metadata.npz")
print(f"  preset.json, metrics.json")
print(f"\nUpload {FINAL_OUT_DIR.name}/ to Kaggle and re-run the inference notebook.")

## Stage 3 — Heads A, B, C

- Toggle **`USE_MIX_TRAIN`** to train on mixed embedding cache.
- Validation always uses **`soundscape_val_embeddings_emb1024.npz`** (clean BirdNET windows).
- Metric: **macro ROC-AUC** over species with **≥3 positives** in the aligned validation matrix (same spirit as your validation notebook).

In [7]:
USE_MIX_TRAIN = True

import json
import pickle

AUTOSAVE_ROOT = PROJECT_ROOT / "notebooks" / "submission_birdnet"
AUTOSAVE_RUN_NAME = "submission_augmented_birdnet"
AUTOSAVE_DIR = AUTOSAVE_ROOT / AUTOSAVE_RUN_NAME
AUTOSAVE_DIR.mkdir(parents=True, exist_ok=True)
AUTOSAVE_METRICS_PATH = AUTOSAVE_DIR / "metrics_stage3.json"

def _save_stage3_metrics(**kwargs):
    payload = {}
    if AUTOSAVE_METRICS_PATH.is_file():
        with open(AUTOSAVE_METRICS_PATH, "r", encoding="utf-8") as f:
            payload = json.load(f)
    payload.update(kwargs)
    with open(AUTOSAVE_METRICS_PATH, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)

def macro_auc_ge3(y_true: np.ndarray, y_score: np.ndarray) -> tuple[float, int, int]:
    pos = y_true.sum(axis=0)
    keep = pos >= 3
    if not np.any(keep):
        return float("nan"), 0, int(np.sum(pos > 0))
    yt = y_true[:, keep]
    ys = y_score[:, keep]
    usable = []
    for j in range(yt.shape[1]):
        col = yt[:, j]
        if col.min() == 0 and col.max() == 1:
            usable.append(j)
    if not usable:
        return float("nan"), int(np.sum(keep)), int(np.sum(pos > 0))
    usable = np.array(usable, dtype=int)
    auc = roc_auc_score(yt[:, usable], ys[:, usable], average="macro")
    return float(auc), int(np.sum(keep)), int(np.sum(pos > 0))


dval = np.load(NPZ_VAL, allow_pickle=True)
X_val, y_val = dval["X_val"].astype(np.float32), dval["y_val"].astype(np.float32)

train_npz = NPZ_MIX if USE_MIX_TRAIN and NPZ_MIX.is_file() else NPZ_CLEAN
dtr = np.load(train_npz, allow_pickle=True)
X_train, y_train = dtr["X_train"].astype(np.float32), dtr["y_train"].astype(np.float32)
print("Train:", train_npz.name, X_train.shape, "| Val:", X_val.shape)
_save_stage3_metrics(train_npz=str(train_npz), x_train_shape=list(X_train.shape), x_val_shape=list(X_val.shape))

Train: train_embeddings_mix_emb1024.npz (106647, 1024) | Val: (739, 1024)


In [17]:
# --- Head A: OvR logistic regression ---
# Some species may be absent in the current training cache (all-zero column),
# which makes binary logistic regression fail. Train only valid columns and
# then scatter back into the full species matrix.
valid_cols_a = np.where((y_train.min(axis=0) < y_train.max(axis=0)))[0]
logreg_valid_cols = valid_cols_a.astype(np.int32)
pred_a = np.zeros((len(X_val), n_species), dtype=np.float32)
if len(valid_cols_a) == 0:
    print("Head A skipped: no trainable columns with both classes.")
    auc_a, k3_a, _ = macro_auc_ge3(y_val, pred_a)
else:
    logreg = MultiOutputClassifier(
        LogisticRegression(max_iter=300, class_weight="balanced", solver="lbfgs", n_jobs=1),
        n_jobs=1,
    )
    logreg.fit(X_train, y_train[:, valid_cols_a])
    pred_a_valid = np.stack([est.predict_proba(X_val)[:, 1] for est in logreg.estimators_], axis=1)
    pred_a[:, valid_cols_a] = pred_a_valid.astype(np.float32)
    auc_a, k3_a, _ = macro_auc_ge3(y_val, pred_a)
print(f"Head A macro-AUC (≥3 pos): {auc_a:.5f} | species_kept≥3: {k3_a} | trained_cols: {len(valid_cols_a)}")

head_a_path = AUTOSAVE_DIR / "head_a_logreg.pkl"
if len(valid_cols_a) > 0:
    with open(head_a_path, "wb") as f:
        pickle.dump(logreg, f)
    head_a_path_str = str(head_a_path)
    print("Saved Head A:", head_a_path)
else:
    head_a_path_str = ""
    print("Head A model not saved (no trainable columns).")
_save_stage3_metrics(
    auc_a=float(auc_a),
    k3_a=int(k3_a),
    logreg_valid_cols=[int(x) for x in logreg_valid_cols.tolist()],
    head_a_path=head_a_path_str,
)

# Head B and Head C are split into separate cells below so they can be run independently.
# This cell intentionally stops after Head A.

/Users/enricozafiris/Desktop/Catolica/T4 Advanced Predictive Analytics/APA-Project/.venv311/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/Users/enricozafiris/Desktop/Catolica/T4 Advanced Predictive Analytics/APA-Project/.venv311/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/Users/enricozafiris/Desktop/Catolica/T4 Advanced Predictive Analytics/APA-Project/.venv311/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=1', please leave it unspecified.
  warnings.warn(msg, category=Futu

KeyboardInterrupt: 

In [16]:
# --- Head B: MLP + per-class positive weights (clip 1..25) ---
import time

class HeadBProgressLogger(tf.keras.callbacks.Callback):
    """Only logs once per epoch."""

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        loss = logs.get("loss", None)
        val_loss = logs.get("val_loss", None)

        loss_s = f"{loss:.5f}" if loss is not None else "n/a"
        val_loss_s = f"{val_loss:.5f}" if val_loss is not None else "n/a"

        print(f"[HeadB] epoch {epoch+1} | loss={loss_s} | val_loss={val_loss_s}")


pos = y_train.sum(axis=0).astype(np.float64)
neg = len(y_train) - pos
pos_weight = np.clip(neg / np.maximum(pos, 1.0), 1.0, 25.0).astype(np.float32)
pw_v = tf.constant(pos_weight)[tf.newaxis, :]


def weighted_multilabel_bce(y_true, y_pred):
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
    return tf.reduce_mean(
        pw_v * y_true * (-tf.math.log(y_pred))
        + (1.0 - y_true) * (-tf.math.log(1.0 - y_pred)),
        axis=-1,
    )


tf.keras.utils.set_random_seed(42)

inp = tf.keras.layers.Input(shape=(EMB_DIM,))
h = tf.keras.layers.Dense(512, activation="relu")(inp)
h = tf.keras.layers.Dropout(0.3)(h)
out = tf.keras.layers.Dense(n_species, activation="sigmoid")(h)

mlp_b = tf.keras.Model(inp, out)
mlp_b.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=weighted_multilabel_bce,
    run_eagerly=True,
)

cb = tf.keras.callbacks.EarlyStopping(
    patience=20, restore_best_weights=True, monitor="val_loss"
)

_progress = HeadBProgressLogger()

mlp_b.fit(
    X_train,
    y_train,
    validation_split=0.1,
    epochs=80,
    batch_size=64,
    callbacks=[_progress, cb],
    verbose=0,  # <- no default Keras prints
)

pred_b = mlp_b.predict(X_val, batch_size=512, verbose=0)
auc_b, k3_b, _ = macro_auc_ge3(y_val, pred_b)

print(f"Head B macro-AUC (≥3 pos): {auc_b:.5f} | species_kept≥3: {k3_b}")

head_b_path = AUTOSAVE_DIR / "head_b_mlp.keras"
mlp_b.save(head_b_path)

_save_stage3_metrics(
    auc_b=float(auc_b),
    k3_b=int(k3_b),
    head_b_path=str(head_b_path),
)

print("Saved Head B:", head_b_path)

[HeadB] epoch 1 | loss=0.09889 | val_loss=1.30531
[HeadB] epoch 2 | loss=0.05493 | val_loss=1.48400
[HeadB] epoch 3 | loss=0.04279 | val_loss=1.57713
[HeadB] epoch 4 | loss=0.03517 | val_loss=1.62457
[HeadB] epoch 5 | loss=0.03005 | val_loss=1.64600
[HeadB] epoch 6 | loss=0.02650 | val_loss=1.66923
[HeadB] epoch 7 | loss=0.02378 | val_loss=1.66561
[HeadB] epoch 8 | loss=0.02149 | val_loss=1.68762
[HeadB] epoch 9 | loss=0.01994 | val_loss=1.68286
[HeadB] epoch 10 | loss=0.01860 | val_loss=1.68870
[HeadB] epoch 11 | loss=0.01756 | val_loss=1.69250
[HeadB] epoch 12 | loss=0.01665 | val_loss=1.68961
[HeadB] epoch 13 | loss=0.01562 | val_loss=1.69276
[HeadB] epoch 14 | loss=0.01517 | val_loss=1.69214
[HeadB] epoch 15 | loss=0.01449 | val_loss=1.68937
[HeadB] epoch 16 | loss=0.01395 | val_loss=1.69236
[HeadB] epoch 17 | loss=0.01379 | val_loss=1.69444
[HeadB] epoch 18 | loss=0.01313 | val_loss=1.69230
[HeadB] epoch 19 | loss=0.01257 | val_loss=1.69227
[HeadB] epoch 20 | loss=0.01256 | val_lo

## Stage 3b (optional) — Normalized feature retry for B/C

If Heads B/C collapse to ~0.5 AUC, run this step to standardize BirdNET features (z-score per dimension) and retrain B/C with plain BCE.

Artifacts are saved separately so you can compare against the original heads without overwriting them.

In [10]:
# --- Head C: same architecture, train with feature-space mixup (Beta 0.4) ---
# Requires weighted_multilabel_bce from the Head B cell.

def train_mlp_mixup_numpy(model, X, y, epochs=80, batch_size=256, lr=1e-3, alpha=0.4):
    opt = tf.keras.optimizers.Adam(learning_rate=lr)
    n = len(X)
    idx = np.arange(n)

    @tf.function
    def train_step(xb, yb):
        with tf.GradientTape() as tape:
            preds = model(xb, training=True)
            loss = tf.reduce_mean(weighted_multilabel_bce(yb, preds))
        grads = tape.gradient(loss, model.trainable_variables)
        opt.apply_gradients(zip(grads, model.trainable_variables))
        return loss

    for ep in range(epochs):
        RNG.shuffle(idx)
        losses = []
        for start in range(0, n, batch_size):
            sel = idx[start : start + batch_size]
            xb = X[sel].copy()
            yb = y[sel].copy()
            if len(xb) > 1:
                lam = RNG.beta(alpha, alpha, size=(len(xb), 1)).astype(np.float32)
                lam = np.maximum(lam, 1.0 - lam)
                perm = RNG.permutation(len(xb))
                xb = lam * xb + (1.0 - lam) * xb[perm]
                yb = lam * yb + (1.0 - lam) * yb[perm]
            loss = train_step(tf.constant(xb), tf.constant(yb))
            losses.append(float(loss.numpy()))
        if (ep + 1) % 10 == 0:
            print(f"  mixup epoch {ep+1} loss={np.mean(losses):.5f}")


inp_c = tf.keras.layers.Input(shape=(EMB_DIM,))
h_c = tf.keras.layers.Dense(512, activation="relu")(inp_c)
h_c = tf.keras.layers.Dropout(0.3)(h_c)
out_c = tf.keras.layers.Dense(n_species, activation="sigmoid")(h_c)
mlp_c = tf.keras.Model(inp_c, out_c)
mlp_c.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=weighted_multilabel_bce)
train_mlp_mixup_numpy(mlp_c, X_train, y_train, epochs=80, batch_size=256)
pred_c = mlp_c.predict(X_val, batch_size=512, verbose=0)
auc_c, k3_c, _ = macro_auc_ge3(y_val, pred_c)
print(f"Head C macro-AUC (≥3 pos): {auc_c:.5f} | species_kept≥3: {k3_c}")

head_c_path = AUTOSAVE_DIR / "head_c_mlp_mixup.keras"
mlp_c.save(head_c_path)
_save_stage3_metrics(auc_c=float(auc_c), k3_c=int(k3_c), head_c_path=str(head_c_path))
print("Saved Head C:", head_c_path)

NameError: name 'weighted_multilabel_bce' is not defined

In [11]:
from sklearn.preprocessing import StandardScaler

# 1) Standardize BirdNET feature dimensions using train split stats.
scaler = StandardScaler(with_mean=True, with_std=True)
X_train_z = scaler.fit_transform(X_train).astype(np.float32)
X_val_z = scaler.transform(X_val).astype(np.float32)

scaler_path = AUTOSAVE_DIR / "feature_scaler_stage3.npz"
np.savez_compressed(
    scaler_path,
    mean=scaler.mean_.astype(np.float32),
    scale=scaler.scale_.astype(np.float32),
)
print("Saved scaler:", scaler_path)

# 2) Head B (normalized features, plain BCE).
tf.keras.utils.set_random_seed(42)
inp_bz = tf.keras.layers.Input(shape=(EMB_DIM,))
h_bz = tf.keras.layers.Dense(512, activation="relu")(inp_bz)
h_bz = tf.keras.layers.Dropout(0.3)(h_bz)
out_bz = tf.keras.layers.Dense(n_species, activation="sigmoid")(h_bz)
mlp_b_norm = tf.keras.Model(inp_bz, out_bz)
mlp_b_norm.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy")
cb_bz = tf.keras.callbacks.EarlyStopping(patience=20, restore_best_weights=True, monitor="val_loss")
mlp_b_norm.fit(
    X_train_z,
    y_train,
    validation_split=0.1,
    epochs=80,
    batch_size=256,
    callbacks=[cb_bz],
    verbose=1,
)
pred_b_norm = mlp_b_norm.predict(X_val_z, batch_size=512, verbose=0)
auc_b_norm, k3_b_norm, _ = macro_auc_ge3(y_val, pred_b_norm)
print(f"Head B (norm) macro-AUC (≥3 pos): {auc_b_norm:.5f} | species_kept≥3: {k3_b_norm}")

head_b_norm_path = AUTOSAVE_DIR / "head_b_mlp_norm.keras"
mlp_b_norm.save(head_b_norm_path)
print("Saved Head B norm:", head_b_norm_path)

# 3) Head C (normalized features + mixup, plain BCE).
def train_mlp_mixup_numpy_plain(model, X, y, epochs=80, batch_size=256, lr=1e-3, alpha=0.4):
    opt = tf.keras.optimizers.Adam(learning_rate=lr)
    n = len(X)
    idx = np.arange(n)

    @tf.function
    def train_step(xb, yb):
        with tf.GradientTape() as tape:
            preds = model(xb, training=True)
            loss = tf.reduce_mean(tf.reduce_mean(tf.keras.losses.binary_crossentropy(yb, preds), axis=-1))
        grads = tape.gradient(loss, model.trainable_variables)
        opt.apply_gradients(zip(grads, model.trainable_variables))
        return loss

    for ep in range(epochs):
        RNG.shuffle(idx)
        losses = []
        for start in range(0, n, batch_size):
            sel = idx[start : start + batch_size]
            xb = X[sel].copy()
            yb = y[sel].copy()
            if len(xb) > 1:
                lam = RNG.beta(alpha, alpha, size=(len(xb), 1)).astype(np.float32)
                lam = np.maximum(lam, 1.0 - lam)
                perm = RNG.permutation(len(xb))
                xb = lam * xb + (1.0 - lam) * xb[perm]
                yb = lam * yb + (1.0 - lam) * yb[perm]
            loss = train_step(tf.constant(xb), tf.constant(yb))
            losses.append(float(loss.numpy()))
        if (ep + 1) % 10 == 0:
            print(f"  mixup(norm) epoch {ep+1} loss={np.mean(losses):.5f}")


inp_cz = tf.keras.layers.Input(shape=(EMB_DIM,))
h_cz = tf.keras.layers.Dense(512, activation="relu")(inp_cz)
h_cz = tf.keras.layers.Dropout(0.3)(h_cz)
out_cz = tf.keras.layers.Dense(n_species, activation="sigmoid")(h_cz)
mlp_c_norm = tf.keras.Model(inp_cz, out_cz)
mlp_c_norm.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy")
train_mlp_mixup_numpy_plain(mlp_c_norm, X_train_z, y_train, epochs=80, batch_size=256)
pred_c_norm = mlp_c_norm.predict(X_val_z, batch_size=512, verbose=0)
auc_c_norm, k3_c_norm, _ = macro_auc_ge3(y_val, pred_c_norm)
print(f"Head C (norm) macro-AUC (≥3 pos): {auc_c_norm:.5f} | species_kept≥3: {k3_c_norm}")

head_c_norm_path = AUTOSAVE_DIR / "head_c_mlp_mixup_norm.keras"
mlp_c_norm.save(head_c_norm_path)
print("Saved Head C norm:", head_c_norm_path)

_save_stage3_metrics(
    scaler_path=str(scaler_path),
    auc_b_norm=float(auc_b_norm),
    k3_b_norm=int(k3_b_norm),
    head_b_norm_path=str(head_b_norm_path),
    auc_c_norm=float(auc_c_norm),
    k3_c_norm=int(k3_c_norm),
    head_c_norm_path=str(head_c_norm_path),
)


Saved scaler: /Users/enricozafiris/Desktop/Catolica/T4 Advanced Predictive Analytics/APA-Project/notebooks/submission_birdnet/submission_augmented_birdnet/feature_scaler_stage3.npz
Epoch 1/80
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0260 - val_loss: 0.0401
Epoch 2/80
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0064 - val_loss: 0.0457
Epoch 3/80
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0050 - val_loss: 0.0505
Epoch 4/80
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0041 - val_loss: 0.0549
Epoch 5/80
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0034 - val_loss: 0.0594
Epoch 6/80
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0029 - val_loss: 0.0639
Epoch 7/80
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0025 - val_loss: 0.0678
Epoch 8/80
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0022 - val_loss: 0.0722
Epoch 9/80
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0019 - val_loss: 0.0760
Epoch 10/80
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 5

KeyboardInterrupt: 

## EXTRA: Add On Stage 2 - Pseudo Labeling of Training Soundscapes (after head was trained)

In [8]:
# ─────────────────────────────────────────────────────────────────────────
# Stage 6a — Pseudo-labeling: embed unlabeled soundscapes + filter
# (with explicit BirdNET preload + per-step timing)
# ─────────────────────────────────────────────────────────────────────────
import time
import json
import gc

PSEUDO_TOP1_THRESHOLD = 0.50
PSEUDO_RUNNERUP_MAX   = 0.40
PSEUDO_WINDOW_SEC = 5.0
PSEUDO_FILE_SEC = 60.0
N_WINDOWS_PER_FILE = int(PSEUDO_FILE_SEC // PSEUDO_WINDOW_SEC)

PSEUDO_DIR = CACHE_DIR
NPZ_PSEUDO = PSEUDO_DIR / "pseudo_labels_emb1024.npz"
PSEUDO_LOG = PSEUDO_DIR / "pseudo_labels_log.json"

# ── STEP 1: Pre-initialize BirdNET (explicit, so we see timing) ──────────
print("Step 1: initializing BirdNET encoder (this takes 30-120s on first call)...", flush=True)
t = time.time()
_init_birdnet_encoder()
print(f"  ✓ BirdNET init: {time.time()-t:.1f}s", flush=True)

# ── STEP 2: Warm up with a dummy embed ───────────────────────────────────
print("\nStep 2: warmup embed call (first real call traces the TF graph)...", flush=True)
t = time.time()
dummy = np.random.randn(int(CLIP_LOAD_SEC * SR_LOAD)).astype(np.float32)
_ = embed_audio_mono(dummy, SR_LOAD)
print(f"  ✓ Warmup embed: {time.time()-t:.1f}s", flush=True)

# ── STEP 3: Load Head B weights ──────────────────────────────────────────
print("\nStep 3: loading Head B weights...", flush=True)
WEIGHTS_PATH = AUTOSAVE_DIR / "head_b_weights.weights.h5"
print(f"  Path: {WEIGHTS_PATH}", flush=True)
print(f"  exists: {WEIGHTS_PATH.is_file()}", flush=True)

if not WEIGHTS_PATH.is_file():
    print("  Creating weights file from .keras model...", flush=True)
    _tmp = tf.keras.models.load_model(AUTOSAVE_DIR / "head_b_mlp.keras", compile=False)
    _tmp.save_weights(WEIGHTS_PATH)
    del _tmp
    gc.collect()

inp = tf.keras.layers.Input(shape=(EMB_DIM,))
h = tf.keras.layers.Dense(512, activation="relu")(inp)
h = tf.keras.layers.Dropout(0.3)(h)
out = tf.keras.layers.Dense(n_species, activation="sigmoid")(h)
head_b_current = tf.keras.Model(inp, out)
head_b_current.load_weights(WEIGHTS_PATH)
print(f"  ✓ Head B loaded ({head_b_current.count_params():,} params)", flush=True)

test_in = np.random.randn(2, EMB_DIM).astype(np.float32)
test_out = head_b_current(test_in, training=False).numpy()
print(f"  sanity: pred range [{test_out.min():.3f}, {test_out.max():.3f}], "
      f"std={test_out.std():.3f}", flush=True)
assert test_out.std() > 0.01

# ── STEP 4: Build unlabeled file list ────────────────────────────────────
all_soundscapes = sorted(TRAIN_SOUNDSCAPES_DIR.glob("*.ogg"))
labeled_stems_set = set(labeled_stems)
unlabeled_files = [p for p in all_soundscapes if p.stem not in labeled_stems_set]
print(f"\nStep 4: {len(unlabeled_files)} unlabeled soundscapes", flush=True)

# ── Window helper ────────────────────────────────────────────────────────
def soundscape_windows_list(file_path, sr=SR_LOAD,
                              win_sec=PSEUDO_WINDOW_SEC, file_sec=PSEUDO_FILE_SEC):
    try:
        y_full, _ = librosa.load(str(file_path), sr=sr, mono=True, duration=file_sec)
    except Exception as e:
        print(f"    [load fail] {file_path.name}: {type(e).__name__}", flush=True)
        return []
    n = int(win_sec * sr)
    n_full = int(file_sec * sr)
    if len(y_full) < n_full:
        y_full = np.pad(y_full, (0, n_full - len(y_full)))
    out_list = []
    for w in range(N_WINDOWS_PER_FILE):
        seg = y_full[w * n: (w + 1) * n]
        if len(seg) < n:
            seg = np.pad(seg, (0, n - len(seg)))
        out_list.append(((w + 1) * int(win_sec), seg.astype(np.float32)))
    return out_list

# ── STEP 5: Time first 3 real files ──────────────────────────────────────
print("\nStep 5: timing first 3 files (should each take 5-15s on M1)...", flush=True)
for i in range(3):
    t = time.time()
    windows = soundscape_windows_list(unlabeled_files[i])
    if not windows:
        print(f"  File {i+1}: failed to load, skipping", flush=True)
        continue
    # KEY OPTIMIZATION: send all 12 windows to encoder as ONE batch
    raw_audio = [seg for _, seg in windows]
    embs = embed_audio_batch(raw_audio, SR_LOAD)
    probs = head_b_current(embs, training=False).numpy()
    dt = time.time() - t
    print(f"  File {i+1}: 12 windows in {dt:.1f}s ({dt/12:.2f}s/window)", flush=True)

print(f"\n  ✓ Pipeline working. Starting full pass...", flush=True)

# ── STEP 6: Full pass with batch embedding per file ──────────────────────
print("\n── Full pass ──", flush=True)
kept_X, kept_y, kept_meta = [], [], []
n_total = 0
n_kept = 0
n_failed = 0
species_count = np.zeros(n_species, dtype=np.int64)

t_start = time.time()
HEARTBEAT_EVERY = 5

for fi, fp in enumerate(unlabeled_files):
    file_t0 = time.time()
    try:
        windows = soundscape_windows_list(fp)
        if not windows:
            n_failed += 1
            continue

        # CRITICAL: batch all 12 windows of this file into ONE embed call
        raw_audio = [seg for _, seg in windows]
        embs = embed_audio_batch(raw_audio, SR_LOAD)
        probs = head_b_current(embs, training=False).numpy()
        n_total += len(windows)

        for i, (end_sec, _) in enumerate(windows):
            p = probs[i]
            top2 = np.argsort(-p)[:2]
            top1_prob = float(p[top2[0]])
            top2_prob = float(p[top2[1]])
            if top1_prob >= PSEUDO_TOP1_THRESHOLD and top2_prob < PSEUDO_RUNNERUP_MAX:
                lbl = np.zeros(n_species, dtype=np.float32)
                lbl[top2[0]] = 1.0
                kept_X.append(embs[i].copy())
                kept_y.append(lbl)
                kept_meta.append({
                    "stem": fp.stem, "end_sec": int(end_sec),
                    "species_idx": int(top2[0]),
                    "top1_prob": top1_prob, "top2_prob": top2_prob,
                })
                species_count[top2[0]] += 1
                n_kept += 1

        del windows, raw_audio, embs, probs

    except Exception as e:
        print(f"  [FILE FAIL {fp.name}]: {type(e).__name__}: {e}", flush=True)
        n_failed += 1
        continue

    file_dt = time.time() - file_t0

    if (fi + 1) % HEARTBEAT_EVERY == 0 or file_dt > 30 or (fi + 1) == len(unlabeled_files):
        elapsed = time.time() - t_start
        rate = (fi + 1) / max(elapsed, 1e-3)
        eta_min = (len(unlabeled_files) - fi - 1) / max(rate, 1e-3) / 60
        kr = 100 * n_kept / max(n_total, 1)
        print(f"  [{fi+1}/{len(unlabeled_files)}] "
              f"kept={n_kept}/{n_total} ({kr:.1f}%) "
              f"| failed={n_failed} | {rate:.2f} f/s | ETA {eta_min:.1f}m "
              f"| last={file_dt:.1f}s", flush=True)

    # Periodic save every 500 files
    if (fi + 1) % 500 == 0 and kept_X:
        try:
            np.savez_compressed(
                str(NPZ_PSEUDO).replace(".npz", "_partial.npz"),
                X_pseudo=np.stack(kept_X).astype(np.float32),
                y_pseudo=np.stack(kept_y).astype(np.float32),
                meta=np.array(kept_meta, dtype=object),
            )
            print(f"    [partial save: {len(kept_X)} pseudo-labels]", flush=True)
        except Exception as e:
            print(f"    [partial save failed: {e}]", flush=True)

    if (fi + 1) % 100 == 0:
        gc.collect()

print(f"\nDone. n_kept={n_kept}, n_total={n_total}, n_failed_files={n_failed}", flush=True)

if not kept_X:
    raise RuntimeError("No pseudo-labels passed the filter.")

X_pseudo = np.stack(kept_X).astype(np.float32)
y_pseudo = np.stack(kept_y).astype(np.float32)
meta_arr = np.array(kept_meta, dtype=object)

np.savez_compressed(NPZ_PSEUDO, X_pseudo=X_pseudo, y_pseudo=y_pseudo, meta=meta_arr)
print(f"Saved: {NPZ_PSEUDO}  X={X_pseudo.shape}", flush=True)

log = {
    "n_files": len(unlabeled_files),
    "n_files_failed": int(n_failed),
    "n_windows_total": int(n_total),
    "n_kept": int(n_kept),
    "keep_rate": float(n_kept / max(n_total, 1)),
    "n_species_with_pseudo": int((species_count > 0).sum()),
    "min_per_species": int(species_count[species_count > 0].min()
                              if (species_count > 0).any() else 0),
    "max_per_species": int(species_count.max()),
    "median_per_species": int(np.median(species_count[species_count > 0])
                                  if (species_count > 0).any() else 0),
    "thresholds": {"top1": PSEUDO_TOP1_THRESHOLD, "runnerup_max": PSEUDO_RUNNERUP_MAX},
}
PSEUDO_LOG.write_text(json.dumps(log, indent=2))
print(f"\nLog: {log}", flush=True)

Step 1: initializing BirdNET encoder (this takes 30-120s on first call)...
BirdNET encoder ready | sample_rate= 48000 | emb_dim= 1024 | batch_size= 32 | workers= 4 | producers= 2
  ✓ BirdNET init: 0.1s

Step 2: warmup embed call (first real call traces the TF graph)...
  ✓ Warmup embed: 5.7s

Step 3: loading Head B weights...
  Path: /Users/enricozafiris/Desktop/Catolica/T4 Advanced Predictive Analytics/APA-Project/notebooks/submission_birdnet/submission_augmented_birdnet/head_b_weights.weights.h5
  exists: True
  ✓ Head B loaded (644,842 params)
  sanity: pred range [0.000, 0.559], std=0.044

Step 4: 10592 unlabeled soundscapes

Step 5: timing first 3 files (should each take 5-15s on M1)...
  File 1: 12 windows in 1.5s (0.13s/window)
  File 2: 12 windows in 1.4s (0.12s/window)
  File 3: 12 windows in 1.3s (0.11s/window)

  ✓ Pipeline working. Starting full pass...

── Full pass ──
  [5/10592] kept=12/60 (20.0%) | failed=0 | 0.72 f/s | ETA 244.7m | last=1.3s
  [10/10592] kept=24/120 (2

In [9]:
# ─────────────────────────────────────────────────────────────────────────
# Stage 6b — Retrain Head B on focal + pseudo-labeled soundscapes
# ─────────────────────────────────────────────────────────────────────────
# Loads:
#   - Focal mix cache from Stage 2b (your existing training data)
#   - Pseudo cache from Stage 6a
# Trains a fresh Head B with sample weights:
#   - focal samples weight = 1.0
#   - pseudo samples weight = 0.5
# Validates on labeled soundscape pool.
# Saves new bundle for submission.

PSEUDO_WEIGHT = 0.5
RETRAIN_EPOCHS = 80
RETRAIN_PATIENCE = 5
RETRAIN_BATCH_SIZE = 256

PSEUDO_BUNDLE = AUTOSAVE_ROOT / "submission_pseudo_birdnet"
PSEUDO_BUNDLE.mkdir(parents=True, exist_ok=True)

# ── Load both caches ─────────────────────────────────────────────────────
print("Loading focal training cache...")
d_focal = np.load(NPZ_MIX, allow_pickle=True)
X_focal, y_focal = d_focal["X_train"].astype(np.float32), d_focal["y_train"].astype(np.float32)
print(f"  focal: X={X_focal.shape}  y={y_focal.shape}")

print("Loading pseudo cache...")
d_pseudo = np.load(NPZ_PSEUDO, allow_pickle=True)
X_pseudo, y_pseudo = d_pseudo["X_pseudo"].astype(np.float32), d_pseudo["y_pseudo"].astype(np.float32)
print(f"  pseudo: X={X_pseudo.shape}  y={y_pseudo.shape}")

# ── Combine + build sample weights ───────────────────────────────────────
X_all = np.concatenate([X_focal, X_pseudo], axis=0)
y_all = np.concatenate([y_focal, y_pseudo], axis=0)
sw_all = np.concatenate([
    np.ones(len(X_focal), dtype=np.float32),
    np.full(len(X_pseudo), PSEUDO_WEIGHT, dtype=np.float32),
], axis=0)
print(f"\nCombined: X={X_all.shape}  y={y_all.shape}  "
      f"sample_weights: focal=1.0 (n={len(X_focal)}), pseudo={PSEUDO_WEIGHT} (n={len(X_pseudo)})")

# Shuffle
shuffle_rng = np.random.default_rng(42)
perm = shuffle_rng.permutation(len(X_all))
X_all, y_all, sw_all = X_all[perm], y_all[perm], sw_all[perm]

# ── Per-class positive weights (uses combined data, but ignores sample weights here) ──
pos = y_all.sum(axis=0).astype(np.float64)
neg = len(y_all) - pos
pw = np.clip(neg / np.maximum(pos, 1.0), 1.0, 25.0).astype(np.float32)
pw_t = tf.constant(pw)[tf.newaxis, :]

def weighted_multilabel_bce_v2(y_true, y_pred):
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
    # Per-sample loss: mean over classes, with class-pos weighting
    # (Sample weighting is applied externally via Keras.fit's sample_weight arg)
    return tf.reduce_mean(
        pw_t * y_true * (-tf.math.log(y_pred))
        + (1.0 - y_true) * (-tf.math.log(1.0 - y_pred)),
        axis=-1,
    )

# ── Build + train fresh Head B ───────────────────────────────────────────
tf.keras.utils.set_random_seed(42)
inp = tf.keras.layers.Input(shape=(EMB_DIM,))
h = tf.keras.layers.Dense(512, activation="relu")(inp)
h = tf.keras.layers.Dropout(0.3)(h)
out = tf.keras.layers.Dense(n_species, activation="sigmoid")(h)
head_b_pseudo = tf.keras.Model(inp, out)
head_b_pseudo.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                         loss=weighted_multilabel_bce_v2)

print(f"\nTraining Head B (with pseudo-labels): {len(X_all)} samples, max {RETRAIN_EPOCHS} epochs")
cb = tf.keras.callbacks.EarlyStopping(
    patience=RETRAIN_PATIENCE, restore_best_weights=True, monitor="val_loss",
)
t0 = time.time()
hist = head_b_pseudo.fit(
    X_all, y_all,
    sample_weight=sw_all,
    validation_split=0.1,
    epochs=RETRAIN_EPOCHS,
    batch_size=RETRAIN_BATCH_SIZE,
    callbacks=[cb],
    verbose=1,
)
print(f"Train time: {time.time()-t0:.1f}s | epochs run: {len(hist.history['loss'])}")

# ── Evaluate on labeled soundscape val ───────────────────────────────────
dval = np.load(NPZ_VAL, allow_pickle=True)
X_val_eval = dval["X_val"].astype(np.float32)
y_val_eval = dval["y_val"].astype(np.float32)
pred_val = head_b_pseudo.predict(X_val_eval, batch_size=512, verbose=0)
auc_pseudo, k3_pseudo, k1_pseudo = macro_auc_ge3(y_val_eval, pred_val)

# Compare to current head's score on same val
pred_val_old = head_b_current.predict(X_val_eval, batch_size=512, verbose=0)
auc_old, k3_old, _ = macro_auc_ge3(y_val_eval, pred_val_old)

print("\n" + "=" * 60)
print("RESULT")
print("=" * 60)
print(f"  Old Head B (focal only):      AUC>=3 = {auc_old:.4f}  (k3={k3_old})")
print(f"  New Head B (focal + pseudo):  AUC>=3 = {auc_pseudo:.4f}  (k3={k3_pseudo})")
print(f"  Delta: {auc_pseudo - auc_old:+.4f}")

# ── Save submission bundle ───────────────────────────────────────────────
head_b_pseudo.save(PSEUDO_BUNDLE / "head_b_mlp.keras")
head_b_pseudo.save_weights(PSEUDO_BUNDLE / "head_b_weights.weights.h5")

# Copy metadata from existing bundle
src_meta = AUTOSAVE_DIR / "metadata.npz"
dst_meta = PSEUDO_BUNDLE / "metadata.npz"
if src_meta.is_file():
    import shutil
    shutil.copy2(src_meta, dst_meta)
else:
    np.savez_compressed(
        dst_meta,
        species_cols=np.array(species_cols, dtype=object),
        emb_dim=np.int32(EMB_DIM),
        sr_load=np.int32(SR_LOAD),
        birdnet_sr=np.int32(BIRDNET_SR),
        clip_load_sec=np.float32(CLIP_LOAD_SEC),
        birdnet_chunk_sec=np.float32(BIRDNET_CHUNK_SEC),
    )

(PSEUDO_BUNDLE / "metrics.json").write_text(json.dumps({
    "auc_old": float(auc_old),
    "auc_pseudo": float(auc_pseudo),
    "delta": float(auc_pseudo - auc_old),
    "n_focal": int(len(X_focal)),
    "n_pseudo": int(len(X_pseudo)),
    "pseudo_weight": float(PSEUDO_WEIGHT),
    "top1_threshold": PSEUDO_TOP1_THRESHOLD,
    "runnerup_max": PSEUDO_RUNNERUP_MAX,
}, indent=2))

print(f"\nSaved bundle: {PSEUDO_BUNDLE}")
print(f"  head_b_mlp.keras")
print(f"  head_b_weights.weights.h5")
print(f"  metadata.npz")
print(f"  metrics.json")
print(f"\nUpload {PSEUDO_BUNDLE.name}/ to Kaggle and re-run inference.")

Loading focal training cache...
  focal: X=(106647, 1024)  y=(106647, 234)
Loading pseudo cache...
  pseudo: X=(10912, 1024)  y=(10912, 234)

Combined: X=(117559, 1024)  y=(117559, 234)  sample_weights: focal=1.0 (n=106647), pseudo=0.5 (n=10912)

Training Head B (with pseudo-labels): 117559 samples, max 80 epochs
Epoch 1/80
414/414 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 0.1281 - val_loss: 0.0718
Epoch 2/80
414/414 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0677 - val_loss: 0.0584
Epoch 3/80
414/414 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0553 - val_loss: 0.0518
Epoch 4/80
414/414 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.0474 - val_loss: 0.0465
Epoch 5/80
414/414 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0420 - val_loss: 0.0429
Epoch 6/80
414/414 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0378 - val_loss: 0.0407
Epoch 7/80
414/414 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0345 - val_loss: 0.0387
Epoch 8/80
414/414 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0315 - val_loss: 0.03

## Stage 7 — Export submission bundle (mix-trained)

Saves a reusable artifact folder with:

- `head_a_logreg.pkl` (sklearn OvR model)
- `head_b_mlp.keras` (Keras)
- `head_c_mlp_mixup.keras` (Keras)
- `metadata.npz` (species order + audio constants)
- `kaggle_inference.ipynb` (BirdNET-feature inference notebook)

This stage is intended for your **augmented/mix run** (`USE_MIX_TRAIN=True`).

In [18]:
import json
import pickle
from datetime import datetime

BUNDLE_ROOT = PROJECT_ROOT / "notebooks" / "submission_birdnet"
RUN_NAME = "submission_augmented_birdnet"
BUNDLE_DIR = BUNDLE_ROOT / RUN_NAME
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)

if not USE_MIX_TRAIN:
    print("WARNING: USE_MIX_TRAIN is False. You are exporting models from a clean-training run.")

# Save trained heads.
pickle_path = BUNDLE_DIR / "head_a_logreg.pkl"
with open(pickle_path, "wb") as f:
    pickle.dump(logreg, f)

mlp_b_path = BUNDLE_DIR / "head_b_mlp.keras"
mlp_c_path = BUNDLE_DIR / "head_c_mlp_mixup.keras"
mlp_b.save(mlp_b_path)
mlp_c.save(mlp_c_path)

# Save metadata required for deterministic inference on Kaggle.
meta_path = BUNDLE_DIR / "metadata.npz"
np.savez_compressed(
    meta_path,
    species_cols=np.array(species_cols, dtype=object),
    logreg_valid_cols=logreg_valid_cols,
    emb_dim=np.int32(EMB_DIM),
    sr_load=np.int32(SR_LOAD),
    birdnet_sr=np.int32(BIRDNET_SR),
    clip_load_sec=np.float32(CLIP_LOAD_SEC),
    birdnet_chunk_sec=np.float32(BIRDNET_CHUNK_SEC),
)

# Build a Kaggle inference notebook that matches this BirdNET-feature pipeline.
kaggle_nb = {
    "cells": [
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "# BirdCLEF 2026 — BirdNET embedding + Head B MLP inference\\n",
                "\\n",
                "Offline notebook using only Head B (MLP). Upload bundle as dataset and set MODEL_DIR."
            ],
        },
        {
            "cell_type": "code",
            "metadata": {},
            "execution_count": None,
            "outputs": [],
            "source": [
                "%pip install -q birdnet librosa scikit-learn tensorflow"
            ],
        },
        {
            "cell_type": "code",
            "metadata": {},
            "execution_count": None,
            "outputs": [],
            "source": [
                "import os\\n",
                "from pathlib import Path\\n",
                "\\n",
                "import librosa\\n",
                "import numpy as np\\n",
                "import pandas as pd\\n",
                "import tensorflow as tf\\n",
                "import birdnet\\n",
                "\\n",
                "COMP_DIR = '/kaggle/input/birdclef-2026'\\n",
                "MODEL_DIR = '/kaggle/input/REPLACE_WITH_YOUR_DATASET_FOLDER/submission_augmented_birdnet'\\n",
                "\\n",
                "meta = np.load(os.path.join(MODEL_DIR, 'metadata.npz'), allow_pickle=True)\\n",
                "species_cols = list(meta['species_cols'])\\n",
                "EMB_DIM = int(meta['emb_dim'])\\n",
                "SR_LOAD = int(meta['sr_load'])\\n",
                "BIRDNET_SR = int(meta['birdnet_sr'])\\n",
                "CLIP_LOAD_SEC = float(meta['clip_load_sec'])\\n",
                "BIRDNET_CHUNK_SEC = float(meta['birdnet_chunk_sec'])\\n",
                "\\n",
                "# Load only Head B\\n",
                "mlp_b = tf.keras.models.load_model(os.path.join(MODEL_DIR, 'head_b_mlp.keras'), compile=False)\\n",
                "\\n",
                "bird_model = birdnet.load('acoustic', '2.4', 'tf')\\n",
                "bird_sr = int(bird_model.get_sample_rate())\\n",
                "BIRDNET_BATCH_SIZE = 32\\n",
                "bird_sess = bird_model.encode_session(batch_size=BIRDNET_BATCH_SIZE, prefetch_ratio=2, n_workers=2, n_producers=1)\\n",
                "bird_sess.__enter__()\\n",
                "\\n",
                "def audio_to_birdnet_chunk(y, sr):\\n",
                "    y = np.asarray(y, dtype=np.float32).reshape(-1)\\n",
                "    if sr != BIRDNET_SR:\\n",
                "        y = librosa.resample(y, orig_sr=sr, target_sr=BIRDNET_SR)\\n",
                "    max_keep = int(CLIP_LOAD_SEC * BIRDNET_SR)\\n",
                "    if len(y) > max_keep:\\n",
                "        y = y[:max_keep]\\n",
                "    need = int(BIRDNET_CHUNK_SEC * BIRDNET_SR)\\n",
                "    if len(y) >= need:\\n",
                "        s0 = max(0, (len(y) - need) // 2)\\n",
                "        y = y[s0:s0+need]\\n",
                "    else:\\n",
                "        y = np.pad(y, (0, need - len(y)))\\n",
                "    return y.astype(np.float32)\\n",
                "\\n",
                "def extract_feat(y, sr):\\n",
                "    chunk = audio_to_birdnet_chunk(y, sr).astype(np.float32)\\n",
                "    res = bird_sess.run_arrays([(chunk, bird_sr)])\\n",
                "    feat = np.asarray(res.embeddings[0, 0, :], dtype=np.float32).reshape(-1)\\n",
                "    if feat.shape[0] != EMB_DIM:\\n",
                "        raise RuntimeError(f'Unexpected BirdNET embedding dim {feat.shape}; expected {EMB_DIM}')\\n",
                "    return feat\\n",
                "\\n",
                "def predict_heads(X):\\n",
                "    # Only use Head B\\n",
                "    pb = mlp_b.predict(X, batch_size=512, verbose=0)\\n",
                "    return pb\\n",
            ],
        },
        {
            "cell_type": "code",
            "metadata": {},
            "execution_count": None,
            "outputs": [],
            "source": [
                "sample_sub = pd.read_csv(os.path.join(COMP_DIR, 'sample_submission.csv'))\\n",
                "test_dir = Path(COMP_DIR) / 'test_soundscapes'\\n",
                "\\n",
                "pred_rows = []\\n",
                "for rid in sample_sub['row_id']:\\n",
                "    stem, end_s = rid.rsplit('_', 1)\\n",
                "    end_sec = int(end_s)\\n",
                "    start_sec = max(0, end_sec - int(CLIP_LOAD_SEC))\\n",
                "    fpath = test_dir / f'{stem}.ogg'\\n",
                "    y, _ = librosa.load(str(fpath), sr=SR_LOAD, mono=True, offset=start_sec, duration=CLIP_LOAD_SEC)\\n",
                "    n = int(CLIP_LOAD_SEC * SR_LOAD)\\n",
                "    if len(y) > n:\\n",
                "        y = y[:n]\\n",
                "    elif len(y) < n:\\n",
                "        y = np.pad(y, (0, n - len(y)))\\n",
                "    feat = extract_feat(y, SR_LOAD)[None, :]\\n",
                "    pred = predict_heads(feat)[0]\\n",
                "    row = {'row_id': rid}\\n",
                "    for c, p in zip(species_cols, pred):\\n",
                "        row[c] = float(np.clip(p, 0.0, 1.0))\\n",
                "    pred_rows.append(row)\\n",
                "\\n",
                "sub = pd.DataFrame(pred_rows)\\n",
                "sub = sample_sub[['row_id']].merge(sub, on='row_id', how='left')\\n",
                "sub[species_cols] = sub[species_cols].fillna(0.0).clip(0.0, 1.0)\\n",
                "sub.to_csv('/kaggle/working/submission.csv', index=False)\\n",
                "print('Saved /kaggle/working/submission.csv', sub.shape)"
            ],
        },
    ],
    "metadata": {
        "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
        "language_info": {"name": "python"},
    },
    "nbformat": 4,
    "nbformat_minor": 5,
}

kaggle_nb_path = BUNDLE_DIR / "kaggle_inference.ipynb"
with open(kaggle_nb_path, "w", encoding="utf-8") as f:
    json.dump(kaggle_nb, f, indent=2)

readme_path = BUNDLE_DIR / "README.txt"
readme_path.write_text(
    "BirdNET submission bundle (mix-trained).\\n"
    f"Created: {datetime.now().isoformat(timespec='seconds')}\\n"
    "Files:\\n"
    "- head_a_logreg.pkl\\n"
    "- head_b_mlp.keras\\n"
    "- head_c_mlp_mixup.keras\\n"
    "- metadata.npz\\n"
    "- kaggle_inference.ipynb\\n"
    "\\n"
    "Kaggle usage:\\n"
    "1) Upload this folder as a Kaggle Dataset.\\n"
    "2) Open kaggle_inference.ipynb and set MODEL_DIR.\\n"
    "3) Run all cells to create /kaggle/working/submission.csv.\\n",
    encoding="utf-8",
)

print("Exported bundle:", BUNDLE_DIR)
for p in sorted(BUNDLE_DIR.iterdir()):
    print(" -", p.name)

Exported bundle: /Users/enricozafiris/Desktop/Catolica/T4 Advanced Predictive Analytics/APA-Project/notebooks/submission_birdnet/submission_augmented_birdnet
 - README.txt
 - feature_scaler_stage3.npz
 - head_a_logreg.pkl
 - head_b_mlp.keras
 - head_b_mlp_norm.keras
 - head_c_mlp_mixup.keras
 - head_c_mlp_mixup_norm.keras
 - kaggle_inference.ipynb
 - metadata.npz
 - metrics_stage3.json


## Ensemble (mean probability)

Rank averaging can be added later; start with a simple mean of calibrated-ish sigmoid outputs.

In [ ]:
pred_ens = (pred_a + pred_b + pred_c) / 3.0
auc_e, k3_e, npos = macro_auc_ge3(y_val, pred_ens)
print(f"Ensemble macro-AUC (≥3 pos): {auc_e:.5f} | species_kept≥3: {k3_e} | species_any_pos: {npos}")

pd.DataFrame(
    [
        {"head": "A_logreg", "macro_auc_ge3": auc_a},
        {"head": "B_mlp_weighted", "macro_auc_ge3": auc_b},
        {"head": "C_mlp_mixup", "macro_auc_ge3": auc_c},
        {"head": "mean_ABC", "macro_auc_ge3": auc_e},
    ]
)

## Notes

- **BirdNET downloads:** first `Analyzer()` init may fetch model weights (network required once).
- **RAM:** embedding ~35k clips is manageable; mixing multiplies rows — reduce `N_MIX_VIEWS` or `MAX_FOCAL` first.
- **Alignment:** focal labels are single-positive multi-hot; soundscape validation is true multi-label — macro-AUC subset ≥3 positives reduces noise from ultra-rare classes.
- To iterate faster, train heads on a subset of cached embeddings (`MAX_FOCAL`) before paying full embedding cost.


## Things I did to make the uploaded things workd

In [19]:
import numpy as np
import tensorflow as tf

candidates = [
    "/Users/enricozafiris/Desktop/Catolica/T4 Advanced Predictive Analytics/APA-Project/.venv/lib/python3.9/site-packages/birdnetlib/models/analyzer/BirdNET_GLOBAL_6K_V2.4_MData_Model_V2_FP16.tflite",
    "/Users/enricozafiris/Library/Application Support/birdnet/acoustic-models/v2.4/tf/model-fp32.tflite",
]

for p in candidates:
    intr = tf.lite.Interpreter(model_path=p)
    intr.allocate_tensors()
    in_d  = intr.get_input_details()[0]
    out_d = intr.get_output_details()[0]
    print(f"\n{p}")
    print(f"  input_shape:  {in_d['shape']}")
    print(f"  output_shape: {out_d['shape']}")
    print(f"  output_dim:   {out_d['shape'][-1]}")


/Users/enricozafiris/Desktop/Catolica/T4 Advanced Predictive Analytics/APA-Project/.venv/lib/python3.9/site-packages/birdnetlib/models/analyzer/BirdNET_GLOBAL_6K_V2.4_MData_Model_V2_FP16.tflite
  input_shape:  [1 3]
  output_shape: [   1 6522]
  output_dim:   6522

/Users/enricozafiris/Library/Application Support/birdnet/acoustic-models/v2.4/tf/model-fp32.tflite
  input_shape:  [     1 144000]
  output_shape: [   1 6522]
  output_dim:   6522


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [25]:
import numpy as np
import librosa
import tensorflow as tf

TFLITE_PATH = "/Users/enricozafiris/Library/Application Support/birdnet/acoustic-models/v2.4/tf/model-fp32.tflite"
EMBEDDING_TENSOR_IDX = 545

# Tell TFLite to preserve all intermediate tensors
intr = tf.lite.Interpreter(
    model_path=TFLITE_PATH,
    experimental_preserve_all_tensors=True,
)
intr.allocate_tensors()
in_idx = intr.get_input_details()[0]["index"]

# Load cache and pick a CLEAN sample
cached = np.load("birdnet_cache/train_embeddings_mix_emb1024.npz", allow_pickle=True)
variants = cached["variant"]
clean_indices = np.where(variants == 0)[0]
i = int(clean_indices[0])
print(f"Picking sample index {i} (variant={variants[i]})")

test_path = str(cached["paths"][i])
expected = cached["X_train"][i]
print(f"Cached embedding: norm={np.linalg.norm(expected):.3f}")

# Preprocessing
y, _ = librosa.load(test_path, sr=32000, mono=True)
y = y[:int(5.0 * 32000)]
y = librosa.resample(y, orig_sr=32000, target_sr=48000)
need = int(3.0 * 48000)
s0 = max(0, (len(y) - need) // 2)
chunk = y[s0:s0+need].astype(np.float32)
if len(chunk) < need:
    chunk = np.pad(chunk, (0, need - len(chunk)))
chunk = chunk.reshape(1, -1)

intr.set_tensor(in_idx, chunk)
intr.invoke()

recomputed = intr.get_tensor(EMBEDDING_TENSOR_IDX).reshape(-1)
print(f"Recomputed: norm={np.linalg.norm(recomputed):.3f}")

cos = np.dot(expected, recomputed) / (np.linalg.norm(expected) * np.linalg.norm(recomputed) + 1e-12)
print(f"Cosine similarity: {cos:.6f}")

Picking sample index 0 (variant=0)
Cached embedding: norm=13.594
Recomputed: norm=13.594
Cosine similarity: 1.000000


In [23]:
import numpy as np
cached = np.load("birdnet_cache/train_embeddings_mix_emb1024.npz", allow_pickle=True)
print("Keys in archive:", list(cached.keys()))
print("Shapes:")
for k in cached.keys():
    print(f"  {k}: {cached[k].shape}")

Keys in archive: ['X_train', 'y_train', 'paths', 'variant']
Shapes:
  X_train: (106647, 1024)
  y_train: (106647, 234)
  paths: (106647,)
  variant: (106647,)


In [27]:
import tensorflow as tf
m = tf.keras.models.load_model(
    "submission_birdnet/submission_augmented_birdnet/head_b_mlp.keras",
    compile=False,
)
m.summary()
print()
for i, layer in enumerate(m.layers):
    print(f"{i}: {layer.__class__.__name__}  config={layer.get_config()}")

Model: "functional_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_10 (InputLayer)     │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 512)            │       524,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 234)            │       120,042 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 644,842 (2.46 MB)

 Trainable params: 644,842 (2.46 MB)

 Non-trainable params: 0 (0.00 B)


0: InputLayer  config={'batch_shape': (None, 1024), 'dtype': 'float32', 'sparse': False, 'ragged': False, 'name': 'input_layer_10', 'optional': False}
1: Dense  config={'name': 'dense_20', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None}, 'units': 512, 'activation': 'relu', 'use_bias': True, 'kernel_initializer': {'module': 'keras.initializers', 'class_name': 'GlorotUniform', 'config': {'seed': None}, 'registered_name': None}, 'bias_initializer': {'module': 'keras.initializers', 'class_name': 'Zeros', 'config': {}, 'registered_name': None}, 'kernel_regularizer': None, 'bias_regularizer': None, 'kernel_constraint': None, 'bias_constraint': None, 'quantization_config': None}
2: Dropout  config={'name': 'dropout_10', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None}, 'rate': 0.3, 'seed': None, 'noise_shape': None}
3: Dense 

In [28]:
m.save_weights("head_b_weights.weights.h5")
print("Saved head_b_weights.weights.h5")
import os
print(f"Size: {os.path.getsize('head_b_weights.weights.h5') / 1e6:.2f} MB")

Saved head_b_weights.weights.h5
Size: 2.60 MB
